# 🌍 WIPO Innovation Capabilities Outlook 2026 - Analysis

This notebook analyzes the **WIPO Innovation Capabilities Outlook 2026** data, covering:
1. Data loading and exploration
2. Innovation ecosystem complexity by country
3. Scatter plot: Ease of diversification (replicating WIPO report charts)
4. Ease of diffusion for innovation domains
5. **Chile-focused analysis**: capabilities, untapped potential, and domain breakdowns

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Color palette matching WIPO report style
COLORS = {
    'Africa': '#9DC3E6',
    'Europe': '#2E75B6',
    'Oceania': '#A9D18E',
    'East Asia': '#548235',
    'Northern America': '#BDD7EE',
    'West and Central Asia': '#C5E0B4',
    'Latin America and the Caribbean': '#00B0F0',
}
DIM_COLORS = {
    'Production': '#00B0F0',
    'Entrepreneurial': '#C5E0B4',
    'Technology': '#2E75B6',
    'Science': '#A9D18E',
}
print('✅ Libraries and configuration loaded')

✅ Libraries and configuration loaded


## 1. Data Loading

In [2]:
outputs = pd.read_parquet('data/outputs.parquet')
caps = pd.read_parquet('data/capabilities.parquet')
dens = pd.read_parquet('data/densities.parquet')
pots = pd.read_parquet('data/potentials.parquet')
uc = pd.read_parquet('data/unit_complexities.parquet')
fc = pd.read_parquet('data/field_complexities.parquet')
fields = pd.read_parquet('data/fields.parquet')
units = pd.read_parquet('data/units.parquet')

files_info = {
    'outputs': outputs, 'capabilities': caps, 'densities': dens,
    'potentials': pots, 'unit_complexities': uc, 'field_complexities': fc,
    'fields': fields, 'units': units
}
summary = pd.DataFrame({
    'Rows': {k: v.shape[0] for k, v in files_info.items()},
    'Columns': {k: v.shape[1] for k, v in files_info.items()},
})
print('📊 Dataset summary:')
display(summary)

In [ ]:
print(f"Available periods: {sorted(uc['Period'].unique())}")
print(f"Countries (2023): {units[units['Period']==2023].shape[0]}")
print(f"Innovation fields: {fields.shape[0]}")
print(f"Dimensions: {fields['Dimension Name'].unique().tolist()}")
print(f"\nContinents: {units['Continent'].unique().tolist()}")

## 2. Ecosystem Complexity by Country (2023)

In [ ]:
latest = uc[uc['Period'] == 2023].merge(units[units['Period'] == 2023], on=['Period', 'Unit'])

# Top 20 by ECI
top20 = latest.nlargest(20, 'Ecosystem Complexity Index')[['Unit Name', 'Ecosystem Complexity Index', 'Diversity Share', 'GDP PC', 'Continent']]
top20 = top20.reset_index(drop=True)
top20.index += 1
top20.columns = ['Country', 'ECI', 'Diversity', 'GDP per capita', 'Continent']
display(top20.style.format({'ECI': '{:.2f}', 'Diversity': '{:.1%}', 'GDP per capita': '${:,.0f}'}).bar(subset=['ECI'], color='#2E75B6'))

## 3. Scatter Plot: Ease of Diversification (WIPO Chart Replication)

Replicates **Figure 3.2** from the report: X-axis = *Ease to diversify in complex fields*, Y-axis = *Ease to diversify in fast growing fields*, size = GDP per capita, color = Continent.

In [ ]:
df = latest.copy()
for col in ['Complexity Outlook Index', 'Growth Outlook Index']:
    mn, mx = df[col].min(), df[col].max()
    df[col + ' (norm)'] = (df[col] - mn) / (mx - mn)

# Bubble sizes based on GDP per capita
gdp_valid = df['GDP PC'].dropna()
df['bubble_size'] = df['GDP PC'].apply(lambda x: np.sqrt(x / 1000) * 8 if pd.notna(x) else 30)

fig, ax = plt.subplots(figsize=(14, 10))

# Background quadrants
ax.axhline(0.5, color='#CCCCCC', linewidth=0.8, zorder=0)
ax.axvline(0.5, color='#CCCCCC', linewidth=0.8, zorder=0)
ax.fill_between([0.5, 1.02], 0.5, 1.02, color='#F0F0F0', alpha=0.5, zorder=0)
ax.fill_between([0.5, 1.02], -0.02, 0.5, color='#F5F5F5', alpha=0.3, zorder=0)

for continent, color in COLORS.items():
    mask = df['Continent'] == continent
    subset = df[mask]
    ax.scatter(subset['Complexity Outlook Index (norm)'], subset['Growth Outlook Index (norm)'],
               s=subset['bubble_size'], c=color, alpha=0.75, edgecolors='white', linewidth=0.5,
               label=continent, zorder=3)

# Annotate key countries
labels = ['Chile', 'Argentina', 'Finland', 'Japan', 'Germany', 'New Zealand',
          'Portugal', 'Thailand', 'Indonesia', 'Uganda', 'El Salvador',
          'United States', 'Brazil', 'Mexico', 'Colombia']
for _, row in df.iterrows():
    name = row['Unit Name']
    if any(l in name for l in labels):
        ax.annotate(name.split('(')[0].strip(), (row['Complexity Outlook Index (norm)'], row['Growth Outlook Index (norm)']),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 3), textcoords='offset points')

ax.set_xlabel('Ease to diversify in complex fields →', fontsize=11, fontweight='bold')
ax.set_ylabel('Ease to diversify in fast growing fields ↑', fontsize=11, fontweight='bold')
ax.set_title('Ease of Diversification for Countries, 2023', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# Quadrant labels
ax.text(0.25, 0.52, 'Fast growing fields ↑', fontsize=9, color='#666', ha='center', style='italic')
ax.text(0.75, 0.48, 'High complexity fields →', fontsize=9, color='#666', ha='center', style='italic')

# GDP legend
for gdp_val, label in [(10000, '10K'), (40000, '40K'), (100000, '100K')]:
    ax.scatter([], [], s=np.sqrt(gdp_val/1000)*8, c='gray', alpha=0.5, label=f'GDP per capita: {label}')

ax.legend(loc='upper left', fontsize=8, ncol=2, framealpha=0.9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig('scatter_ease_of_diversification.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: scatter_ease_of_diversification.png')

## 4. Ease of Diffusion by Innovation Domains (Replicates Figure 3.3)

In [ ]:
fc_2023 = fc[fc['Period'] == 2023].merge(fields, on='Field ID')

# Aggregate by domain
domain_agg = fc_2023.groupby(['Domain Name', 'Dimension Name']).agg({
    'Complexity Outlook Index': 'mean',
    'Growth Outlook Index': 'mean',
    'Ubiquity Share': 'mean',
}).reset_index()

# Normalize
for col in ['Complexity Outlook Index', 'Growth Outlook Index']:
    mn, mx = domain_agg[col].min(), domain_agg[col].max()
    domain_agg[col + ' (norm)'] = (domain_agg[col] - mn) / (mx - mn)

fig, ax = plt.subplots(figsize=(12, 9))
ax.axhline(0.5, color='#CCCCCC', linewidth=0.8)
ax.axvline(0.5, color='#CCCCCC', linewidth=0.8)

for dim, color in DIM_COLORS.items():
    mask = domain_agg['Dimension Name'] == dim
    subset = domain_agg[mask]
    ax.scatter(subset['Complexity Outlook Index (norm)'], subset['Growth Outlook Index (norm)'],
               s=120, c=color, alpha=0.8, edgecolors='white', linewidth=1, label=f'{dim} domains', zorder=3)

# Annotate selected domains
for _, row in domain_agg.iterrows():
    if row['Growth Outlook Index (norm)'] > 0.6 or row['Growth Outlook Index (norm)'] < 0.3 or \
       row['Complexity Outlook Index (norm)'] > 0.8 or row['Complexity Outlook Index (norm)'] < 0.2:
        ax.annotate(row['Domain Name'], (row['Complexity Outlook Index (norm)'], row['Growth Outlook Index (norm)']),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 3), textcoords='offset points')

ax.set_xlabel('Ease to diffuse into complex ecosystems →', fontsize=11, fontweight='bold')
ax.set_ylabel('Ease to diffuse into fast growing ecosystems ↑', fontsize=11, fontweight='bold')
ax.set_title('Ease of Diffusion for Innovation Domains, 2023', fontsize=14, fontweight='bold', pad=15)
ax.text(0.25, 0.52, 'Fast growing ecosystems ↑', fontsize=9, color='#666', ha='center', style='italic')
ax.text(0.75, 0.48, 'Complex ecosystems →', fontsize=9, color='#666', ha='center', style='italic')
ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig('scatter_ease_of_diffusion_domains.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: scatter_ease_of_diffusion_domains.png')

## 5. 🇨🇱 Focus Analysis: Chile

In [ ]:
# Chile summary
chile_uc = uc[uc['Unit'] == 'CL'].merge(units[units['Unit'] == 'CL'], on=['Period', 'Unit'])
chile_latest = chile_uc[chile_uc['Period'] == 2023].iloc[0]

print('='*60)
print('🇨🇱 CHILE - Innovation Capabilities Overview 2023')
print('='*60)
print(f"  Ecosystem Complexity Index (ECI): {chile_latest['Ecosystem Complexity Index']:.3f}")
print(f"  Diversity Share:                  {chile_latest['Diversity Share']:.1%}")
print(f"  Complexity Outlook Index:         {chile_latest['Complexity Outlook Index']:.1f}")
print(f"  Growth Outlook Index:             {chile_latest['Growth Outlook Index']:.1f}")
print(f"  GDP per capita (PPP):             ${chile_latest['GDP PC']:,.0f}")
print(f"  Population:                       {chile_latest['Population']:,.0f}")

# Ranking in LATAM
latam = latest[latest['Continent'] == 'Latin America and the Caribbean'].copy()
latam['ECI_rank'] = latam['Ecosystem Complexity Index'].rank(ascending=False)
chile_rank = latam[latam['Unit'] == 'CL']['ECI_rank'].values[0]
print(f"\n  ECI Rank in LATAM:              #{int(chile_rank)} of {len(latam)}")

# Global ranking
latest['ECI_rank_global'] = latest['Ecosystem Complexity Index'].rank(ascending=False)
chile_global = latest[latest['Unit'] == 'CL']['ECI_rank_global'].values[0]
print(f"  Global ECI Rank:                 #{int(chile_global)} of {len(latest)}")

### 5.1 Temporal Evolution of Chile

In [ ]:
chile_ts = uc[uc['Unit'] == 'CL'].sort_values('Period')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics = [
    ('Ecosystem Complexity Index', 'ECI', '#2E75B6'),
    ('Diversity Share', 'Diversity', '#00B0F0'),
    ('Complexity Outlook Index', 'Complexity Outlook', '#548235'),
    ('Growth Outlook Index', 'Growth Outlook', '#A9D18E'),
]
for ax, (col, title, color) in zip(axes.flat, metrics):
    ax.plot(chile_ts['Period'], chile_ts[col], color=color, linewidth=2.5, marker='o', markersize=4)
    ax.fill_between(chile_ts['Period'], chile_ts[col], alpha=0.1, color=color)
    ax.set_title(f'Chile: {title}', fontweight='bold')
    ax.set_xlabel('Year')
    ax.grid(True, alpha=0.2)

fig.suptitle('🇨🇱 Chile - Evolution of Innovation Indicators (2001-2023)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('chile_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Chile's Capabilities by Dimension and Domain

In [ ]:
chile_caps = caps[(caps['Unit'] == 'CL') & (caps['Period'] == 2023)]
chile_with_cap = chile_caps[chile_caps['Innovation Capability (Binary)']].merge(fields, on='Field ID')

# By dimension
dim_counts = chile_with_cap.groupby('Dimension Name').size().sort_values(ascending=True)
total_by_dim = fields.groupby('Dimension Name').size()
dim_pct = (dim_counts / total_by_dim * 100).sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_bars = [DIM_COLORS.get(d, '#999') for d in dim_counts.index]
ax1.barh(dim_counts.index, dim_counts.values, color=colors_bars, edgecolor='white', height=0.6)
for i, (v, p) in enumerate(zip(dim_counts.values, dim_pct.values)):
    ax1.text(v + 2, i, f'{v} ({p:.0f}%)', va='center', fontsize=10, fontweight='bold')
ax1.set_title("Chile's Capabilities by Dimension", fontweight='bold')
ax1.set_xlabel('Number of fields with capability')

# By top domains
dom_counts = chile_with_cap.groupby('Domain Name').size().sort_values(ascending=True).tail(15)
dom_dims = chile_with_cap.groupby('Domain Name')['Dimension Name'].first()
colors_dom = [DIM_COLORS.get(dom_dims.get(d, ''), '#999') for d in dom_counts.index]
ax2.barh(dom_counts.index, dom_counts.values, color=colors_dom, edgecolor='white', height=0.7)
for i, v in enumerate(dom_counts.values):
    ax2.text(v + 0.3, i, str(v), va='center', fontsize=9, fontweight='bold')
ax2.set_title('Top 15 Capable Domains (Chile)', fontweight='bold')
ax2.set_xlabel('Number of fields')

fig.suptitle('🇨🇱 Chile - Innovation Capabilities 2023', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chile_capabilities.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 🇨🇱 Untapped Complex Potential in Chile

Analysis of innovation potential not yet realized by Chile, grouped by output type (dimension).

In [ ]:
# Merge potentials, capabilities, outputs, and field complexities for Chile
chile_pots = pots[(pots['Unit'] == 'CL') & (pots['Period'] == 2023)]
chile_caps_all = caps[(caps['Unit'] == 'CL') & (caps['Period'] == 2023)]
chile_out = outputs[(outputs['Unit'] == 'CL') & (outputs['Period'] == 2023)]
fc_2023_full = fc[fc['Period'] == 2023]

# Build analysis: fields where Chile lacks capability but has high potential
analysis = chile_pots.merge(chile_caps_all[['Field ID', 'Innovation Capability (Binary)']], on='Field ID', how='left')
analysis = analysis.merge(fields, on='Field ID')
analysis = analysis.merge(fc_2023_full[['Field ID', 'Capability Complexity Index']], on='Field ID', how='left')

# Untapped = no capability but positive potential
untapped = analysis[~analysis['Innovation Capability (Binary)']].copy()
complex_untapped = untapped[untapped['Capability Complexity Index'] > 0]

# Summary by dimension
dim_map = {'Technology': 'Complex patents', 'Production': 'Complex exports',
           'Entrepreneurial': 'Complex trademarks', 'Science': 'Complex publications'}

total_fields = fields.groupby('Dimension Name').size()
untapped_complex = complex_untapped.groupby('Dimension Name').size()
untapped_potential = complex_untapped.groupby('Dimension Name')['Potential'].sum()

share_untapped = (untapped_complex / total_fields * 100).fillna(0)

summary_df = pd.DataFrame({
    'Type of outputs': [dim_map.get(d, d) for d in share_untapped.index],
    'Share of untapped complex potential (%)': share_untapped.values,
    'Untapped complex outputs': untapped_complex.values,
    'Total potential': untapped_potential.values,
}).sort_values('Share of untapped complex potential (%)', ascending=False)

display(summary_df.reset_index(drop=True).style.format({
    'Share of untapped complex potential (%)': '{:.0f}%',
    'Total potential': '{:,.1f}',
}).bar(subset=['Share of untapped complex potential (%)'], color='#00B0F0'))

### 5.4 🇨🇱 Bar Chart: Chile's Untapped Potential by Domain

**Chile-filtered chart** showing domains with the highest unrealized innovation potential, colored by innovation dimension.

In [ ]:
# Group untapped potential by domain for Chile
untapped_by_domain = complex_untapped.groupby(['Domain Name', 'Dimension Name']).agg(
    count=('Potential', 'size'),
    total_potential=('Potential', 'sum'),
    avg_complexity=('Capability Complexity Index', 'mean'),
).reset_index().sort_values('total_potential', ascending=True)

# Top 20 domains by potential
top_domains = untapped_by_domain.tail(20)

fig, ax = plt.subplots(figsize=(12, 9))
colors_bar = [DIM_COLORS.get(d, '#999') for d in top_domains['Dimension Name']]
bars = ax.barh(range(len(top_domains)), top_domains['total_potential'], color=colors_bar,
               edgecolor='white', height=0.7, alpha=0.9)

ax.set_yticks(range(len(top_domains)))
ax.set_yticklabels(top_domains['Domain Name'], fontsize=9)

# Value labels
for i, (v, c) in enumerate(zip(top_domains['total_potential'], top_domains['count'])):
    ax.text(v + max(top_domains['total_potential'])*0.01, i, f'{v:,.0f} ({c} fields)', va='center', fontsize=8)

# Legend
legend_elements = [Line2D([0], [0], marker='s', color='w', markerfacecolor=c, markersize=12, label=d)
                   for d, c in DIM_COLORS.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, title='Dimension', title_fontsize=10)

ax.set_xlabel('Total Untapped Potential', fontsize=11, fontweight='bold')
ax.set_title('🇨🇱 Chile: Top 20 Innovation Domains with Highest Untapped Potential (2023)',
             fontsize=13, fontweight='bold', pad=15)
ax.grid(True, axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('chile_untapped_potential_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: chile_untapped_potential_bars.png')

### 5.5 🇨🇱 Chile: Relatedness Density by Domain

In [ ]:
chile_dens = dens[(dens['Unit'] == 'CL') & (dens['Period'] == 2023)].merge(fields, on='Field ID')

# Average density by domain, colored by whether Chile has capability
chile_caps_merged = chile_caps_all.merge(fields, on='Field ID')
cap_by_domain = chile_caps_merged.groupby('Domain Name')['Innovation Capability (Binary)'].mean()
dens_by_domain = chile_dens.groupby(['Domain Name', 'Dimension Name'])['Relatedness Density'].mean().reset_index()
dens_by_domain = dens_by_domain.sort_values('Relatedness Density', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
colors_d = [DIM_COLORS.get(d, '#999') for d in dens_by_domain['Dimension Name']]
ax.barh(range(len(dens_by_domain)), dens_by_domain['Relatedness Density'], color=colors_d,
        edgecolor='white', height=0.7, alpha=0.85)
ax.set_yticks(range(len(dens_by_domain)))
ax.set_yticklabels(dens_by_domain['Domain Name'], fontsize=8)

for i, v in enumerate(dens_by_domain['Relatedness Density']):
    ax.text(v + 0.003, i, f'{v:.2f}', va='center', fontsize=7)

legend_elements = [Line2D([0], [0], marker='s', color='w', markerfacecolor=c, markersize=12, label=d)
                   for d, c in DIM_COLORS.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, title='Dimension')
ax.set_xlabel('Average Relatedness Density', fontsize=11, fontweight='bold')
ax.set_title('🇨🇱 Chile: Relatedness Density by Innovation Domain (2023)',
             fontsize=13, fontweight='bold', pad=15)
ax.grid(True, axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('chile_relatedness_density.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 🇨🇱 Chile vs LATAM: Complexity Comparison

In [ ]:
latam_ts = uc.merge(units[['Period', 'Unit', 'Continent']], on=['Period', 'Unit'])
latam_ts = latam_ts[latam_ts['Continent'] == 'Latin America and the Caribbean']

# Chile vs LATAM average
chile_ts = latam_ts[latam_ts['Unit'] == 'CL'].sort_values('Period')
latam_avg = latam_ts[latam_ts['Unit'] != 'CL'].groupby('Period').mean(numeric_only=True).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title in zip(axes, 
    ['Ecosystem Complexity Index', 'Complexity Outlook Index', 'Growth Outlook Index'],
    ['ECI', 'Complexity Outlook', 'Growth Outlook']):
    ax.plot(chile_ts['Period'], chile_ts[col], color='#00B0F0', linewidth=2.5, marker='o', markersize=3, label='Chile')
    ax.plot(latam_avg['Period'], latam_avg[col], color='#999', linewidth=2, linestyle='--', label='LATAM average')
    ax.fill_between(chile_ts['Period'], chile_ts[col], latam_avg[col], alpha=0.1, color='#00B0F0')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

fig.suptitle('🇨🇱 Chile vs LATAM Average (2001-2023)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chile_vs_latam.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.7 🇨🇱 Strategic Fields: High Complexity + High Potential for Chile

In [ ]:
# Fields Chile doesn't yet have capability in, ranked by potential and complexity
strategic = complex_untapped.nlargest(25, 'Potential')[[
    'Field Name', 'Domain Name', 'Dimension Name', 'Potential', 'Capability Complexity Index'
]].reset_index(drop=True)
strategic.index += 1
strategic.columns = ['Field', 'Domain', 'Dimension', 'Potential', 'Complexity']

print('🎯 Top 25 strategic fields for Chile (high complexity + high potential):')
display(strategic.style.format({'Potential': '{:,.1f}', 'Complexity': '{:.2f}'}).background_gradient(
    subset=['Potential'], cmap='Blues').background_gradient(subset=['Complexity'], cmap='Greens'))

---
## Summary

### Main findings:
1. **Chile leads LATAM** in Complexity and Growth Outlook indices
2. **Science** is Chile's strongest dimension (276 fields with capability)
3. Significant **untapped potential** exists especially in production and technology
4. **Agricultural and Environmental Sciences** domains have the highest relatedness density
5. Chile has shown a **sustained positive trend** in all indicators since 2001